## Setup

In [74]:
import pandas as pd

In [75]:
# Import the wcv collision data csv file into a dataframe
df = pd.read_csv('./datasets/WCV Collision Data.csv')

In [76]:
df.columns

Index(['Organization Name', 'Case Number', 'Patient ID', 'Common Species Name',
       'Date Admitted', 'Circumstances of Rescue', 'Rescue State',
       'Rescue Juristiction', 'Rescue Address', 'Other Rescue Information',
       'Latitude', 'Longitude', 'Elevation', 'Disposition'],
      dtype='object')

In [77]:
# Drop any duplicate rows
df = df.drop_duplicates()

## Data Pipeline
Column by Column

In [78]:
# Validate that the 'Circumstances of Rescue' column only contains the vechile collision reason
assert len(df['Circumstances of Rescue'].unique()) == 1
assert df['Circumstances of Rescue'].unique() == ['Collision / Moving object / Car/truck/motorcycle']

In [79]:
# Drop the circumstaces of rescue column - not needed anymore
df = df.drop(columns=['Circumstances of Rescue'])

In [80]:
# Validate that the rescue state is only VA
assert len(df['Rescue State'].unique()) == 1
assert df['Rescue State'].unique() == ['VA']

In [81]:
# Drop the rescue state column - not needed anymore
df = df.drop(columns=['Rescue State'])

In [82]:
# Drop other unnecessary columns
df = df.drop(columns=['Elevation', 'Other Rescue Information', 'Rescue Address', 'Rescue Juristiction',])

In [83]:
# Remove spaces and replace from remaining column names
df.columns = df.columns.str.replace(' ', '')

In [84]:
df.columns

Index(['OrganizationName', 'CaseNumber', 'PatientID', 'CommonSpeciesName',
       'DateAdmitted', 'Latitude', 'Longitude', 'Disposition'],
      dtype='object')

### CommonSpeciesName Column

In [85]:
# Lowercase CommonSpeciesName
df['CommonSpeciesName'] = df['CommonSpeciesName'].str.lower()

### GeneralSpeciesName Column

In [86]:
collision_mapping = pd.read_csv("../lookup_tables/CollisionAnimalMapping.csv")
# Check if all unique values in CommonSpeciesName are in the collision_mapping
unique_common_names = df["CommonSpeciesName"].unique()
missing_names = set(unique_common_names) - set(collision_mapping["Animal"].str.lower())
assert len(missing_names) == 0, f"Missing common names in collision mapping: {missing_names}"

In [87]:
# Create a column called GeneralSpeciesName based on looking up the CommonSpeciesName in the CollisionAnimalMapping.csv file
collision_mapping["Animal"] = collision_mapping["Animal"].str.lower()
df["GeneralSpeciesName"] = df["CommonSpeciesName"].map(
    dict(zip(collision_mapping["Animal"], collision_mapping["Mapping"]))
)

### Vertebrate Column

In [88]:
animals_categorized = pd.read_csv("../lookup_tables/AnimalsCategorized.csv")
# Check if all unique values in CommonSpeciesName are in the animals_categorized
unique_common_names = df["CommonSpeciesName"].unique()
missing_names = set(unique_common_names) - set(animals_categorized["CommonSpeciesName"].str.lower())
assert len(missing_names) == 0, f"Missing common names in animals categorized: {missing_names}"


In [89]:
# Create a column called Vertebrate based on looking up CommonSpeciesName in the AnimalsCategorized.csv file
animals_categorized["CommonSpeciesName"] = animals_categorized["CommonSpeciesName"].str.lower()
df["Vertebrate"] = df["CommonSpeciesName"].map(
    dict(zip(animals_categorized["CommonSpeciesName"], animals_categorized["Vertebrate"]))
)

### DateAdmittedYear Column

In [90]:
# Fill the "DateAdmittedYear" column with the year from "patients.admitted_at" column

# Convert DateAdmitted to datetime
df["DateAdmitted_DT"] = pd.to_datetime(df["DateAdmitted"], errors='coerce')
df["DateAdmittedYear"] = df["DateAdmitted_DT"].dt.year

In [91]:
# Validate that there are no null values in the DateAdmittedYear column
assert df["DateAdmittedYear"].notnull().all(), "There are null values in the DateAdmittedYear column"

### Season Column

In [92]:
'''
Winter: December 22 - March 20
Spring: March 21 - June 20
Summer: June 21 - September 22
Autumn: September 23 - December 21
'''
# Using the 'patients.admitted_at' column, create a new column called 'Season' based on the date ranges above
def get_season(date):
    if date.month == 12 and date.day >= 22 or date.month in [1, 2] or (date.month == 3 and date.day <= 20):
        return "Winter"
    elif (date.month == 3 and date.day >= 21) or date.month in [4, 5] or (date.month == 6 and date.day <= 20):
        return "Spring"
    elif (date.month == 6 and date.day >= 21) or date.month in [7, 8] or (date.month == 9 and date.day <= 22):
        return "Summer"
    else:
        return "Autumn"
df["Season"] = df["DateAdmitted_DT"].apply(get_season)

In [93]:
# Validate that there are no null seasons in the Season column
assert df["Season"].notnull().all(), "There are null values in the Season column"
# Validate that there are only 4 unique values in the Season column
assert set(df["Season"].unique()) == {"Winter", "Spring", "Summer", "Autumn"}, "There are unexpected values in the Season column"

### DateAdmittedMonth Column

In [94]:
df["DateAdmittedMonth"] = df["DateAdmitted_DT"].dt.month_name()

In [95]:
# Validate that there are no null values in the DateAdmittedMonth column
assert df["DateAdmittedMonth"].notnull().all(), "There are null values in the DateAdmittedMonth column"

### DateAdmittedDOM Column

In [96]:
df["DateAdmittedDOM"] = df["DateAdmitted_DT"].dt.day

In [97]:
# Validate that there are no null values in the DateAdmittedDOM column
assert df["DateAdmittedDOM"].notnull().all(), "There are null values in the DateAdmittedDOM column"

### DateAdmittedDOW Column

In [98]:
df["DateAdmittedDOW"] = df["DateAdmitted_DT"].dt.day_name()

In [99]:
# Validate that there no null values in the DateAdmittedDOW column
assert df["DateAdmittedDOW"].notnull().all(), "There are null values in the DateAdmittedDOW column"

### Disposition Column

In [100]:
df["Disposition"].value_counts()

Disposition
Euthanized      982
Died            340
Released        278
Transferred     119
Active           16
Self-Release      4
Name: count, dtype: int64

In [101]:
# Drop rows where the disposition is "Active", outcomes unknown so we don't want to include them in the analysis
df = df[df["Disposition"] != "Active"]

In [102]:
# Standardize the Disposition values:Active, Died, Released, Transferred

# if the disposition contains the string "died" or "euthanized", set the disposition to "Died"
df.loc[df["Disposition"].str.contains("died|dead", case=False, na=False), "Disposition"] = "Died"
# if the disposition contains the string "release", set the disposition to "Released"
df.loc[df["Disposition"].str.contains("release", case=False, na=False), "Disposition"] = "Released"
# if the disposition contains the string "transferred", set the disposition to "Transferred"
df.loc[df["Disposition"].str.contains("transferred", case=False, na=False), "Disposition"] = "Transferred"
# if the disposition contains the string "euthanized", set the disposition to "Euthanized"
df.loc[df["Disposition"].str.contains("euthanized", case=False, na=False), "Disposition"] = "Euthanized"

In [103]:
# Validate that only the values Active, Died, Released, Transferred are present in the Disposition column
valid_dispositions = {"Euthanized", "Died", "Released", "Transferred"}
assert df["Disposition"].notnull().all(), "There are null values in the Disposition column"
assert set(df["Disposition"].unique()).issubset(valid_dispositions), f"Invalid dispositions found: {set(df['Disposition'].unique()) - valid_dispositions}"

### DayOfWeekNumber

In [104]:
df["DayOfWeekNumber"] = df["DateAdmitted_DT"].dt.dayofweek

In [105]:
# Validate that there are no null values in the DayOfWeekNumber column
assert df["DayOfWeekNumber"].notnull().all(), "There are null values in the DayOfWeekNumber column"

### MonthNumber

In [106]:
df["MonthNumber"] = df["DateAdmitted_DT"].dt.month

In [107]:
# Validate that there are no null values in the MonthNumber column
assert df["MonthNumber"].notnull().all(), "There are null values in the MonthNumber column"

### Lat and Long

In [108]:
# Check for columsn where lat long is null
print(f"There are {len(df[df['Latitude'].isnull() | df['Longitude'].isnull()])} rows with null latitude or longitude")

There are 5 rows with null latitude or longitude


In [109]:
# Drop rows where lat long is null
df = df.dropna(subset=['Latitude', 'Longitude'])

In [110]:
# Validate that there are no empty latitude and longitude values
assert df["Latitude"].notnull().all(), f"There are null values in the Latitude column: {df[df['Latitude'].isnull()]}"
assert df["Longitude"].notnull().all(), f"There are null values in the Longitude column: {df[df['Longitude'].isnull()]}" 

## Output File

In [111]:
cols_to_keep = [
    'OrganizationName',
    'CaseNumber',
    'PatientID',
    'CommonSpeciesName',
    'DateAdmitted',
    'Latitude',
    'Longitude',
    'Disposition',
    'GeneralSpeciesName',
    'Vertebrate',
    'DateAdmittedYear',
    'Season',
    'DateAdmittedMonth',
    'DateAdmittedDOM',
    'DateAdmittedDOW',
    'DayOfWeekNumber',
    'MonthNumber'
    ]

In [112]:
# Get earliest and latest DateAdmitted
earliest_date = df["DateAdmitted_DT"].min()
latest_date = df["DateAdmitted_DT"].max()
print(f"Earliest DateAdmitted: {earliest_date.date()}")
print(f"Latest DateAdmitted: {latest_date.date()}")

Earliest DateAdmitted: 2014-02-09
Latest DateAdmitted: 2023-12-02


In [113]:
output_file = f"./datasets/WCV_transformed_{earliest_date.date()}_to_{latest_date.date()}.csv"
df[cols_to_keep].to_csv(output_file, index=False)